# NLSC ????????????

?? `tainan_hex.csv` ? EPSG:3826 ????????????? NLSC `PHOTO2` ????????????????? `id.png` ???


In [1]:

import io
import math
from pathlib import Path
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw

TILE_SIZE = 256
GRID_CSV_PATH = Path("tainan_hex.csv")
OUTPUT_DIR = Path("satellite_downloads")
WMTS_TEMPLATE = (
    "https://wmts.nlsc.gov.tw/wmts/{layer}/default/GoogleMapsCompatible/{z}/{y}/{x}"
)
HTTP_HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; nlsc-photo-downloader/1.0)"}


In [ ]:
def twd97_to_lonlat(x: float, y: float) -> tuple[float, float]:
    """Convert EPSG:3826 (TWD97 / TM2 zone 121) x/y to lon/lat."""
    a = 6378137.0
    b = 6356752.314245
    lon0 = math.radians(121.0)
    k0 = 0.9999
    dx = 250000.0

    e = math.sqrt(1 - (b * b) / (a * a))
    e2 = e * e
    e1 = (1 - math.sqrt(1 - e2)) / (1 + math.sqrt(1 - e2))
    x = float(x) - dx
    y = float(y)

    m = y / k0
    mu = m / (a * (1 - e2 / 4 - 3 * e2**2 / 64 - 5 * e2**3 / 256))
    j1 = 3 * e1 / 2 - 27 * e1**3 / 32
    j2 = 21 * e1**2 / 16 - 55 * e1**4 / 32
    j3 = 151 * e1**3 / 96
    j4 = 1097 * e1**4 / 512
    fp = mu + j1 * math.sin(2 * mu) + j2 * math.sin(4 * mu) + j3 * math.sin(6 * mu) + j4 * math.sin(8 * mu)

    ep2 = e2 / (1 - e2)
    sin_fp = math.sin(fp)
    cos_fp = math.cos(fp)
    tan_fp = math.tan(fp)
    c1 = ep2 * cos_fp**2
    t1 = tan_fp**2
    r1 = a * (1 - e2) / (1 - e2 * sin_fp**2) ** 1.5
    n1 = a / math.sqrt(1 - e2 * sin_fp**2)
    d = x / (n1 * k0)

    lat = fp - (n1 * tan_fp / r1) * (
        d**2 / 2
        - (5 + 3 * t1 + 10 * c1 - 4 * c1**2 - 9 * ep2) * d**4 / 24
        + (61 + 90 * t1 + 298 * c1 + 45 * t1**2 - 252 * ep2 - 3 * c1**2) * d**6 / 720
    )
    lon = lon0 + (
        d
        - (1 + 2 * t1 + c1) * d**3 / 6
        + (5 - 2 * c1 + 28 * t1 - 3 * c1**2 + 8 * ep2 + 24 * t1**2) * d**5 / 120
    ) / cos_fp

    return math.degrees(lon), math.degrees(lat)


def lonlat_to_pixel(lon: float, lat: float, zoom: int) -> tuple[float, float]:
    lat = max(min(lat, 85.05112878), -85.05112878)
    scale = TILE_SIZE * (2**zoom)
    x = (lon + 180.0) / 360.0 * scale
    lat_rad = math.radians(lat)
    y = (1.0 - math.log(math.tan(lat_rad) + 1.0 / math.cos(lat_rad)) / math.pi) / 2.0 * scale
    return x, y


def fetch_tile(x: int, y: int, z: int, layer: str = "PHOTO2") -> Image.Image:
    url = WMTS_TEMPLATE.format(layer=layer, z=z, y=y, x=x)
    request = Request(url, headers=HTTP_HEADERS)
    with urlopen(request, timeout=20) as response:
        content = response.read()
    return Image.open(io.BytesIO(content)).convert("RGB")


def load_hex_grid(csv_path: str | Path = GRID_CSV_PATH) -> pd.DataFrame:
    grid_df = pd.read_csv(csv_path)
    required_cols = {"id", "X", "Y", "left", "top", "right", "bottom"}
    missing_cols = required_cols - set(grid_df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {sorted(missing_cols)}")
    return grid_df


def get_hex_geometry(row: pd.Series) -> dict:
    grid_id = int(row["id"])
    left = float(row["left"])
    right = float(row["right"])
    top = float(row["top"])
    bottom = float(row["bottom"])
    cx = (left + right) / 2
    cy = (top + bottom) / 2
    w = right - left
    h = top - bottom

    vertices_3826 = [
        (cx - w / 2, cy),
        (cx - w / 4, cy + h / 2),
        (cx + w / 4, cy + h / 2),
        (cx + w / 2, cy),
        (cx + w / 4, cy - h / 2),
        (cx - w / 4, cy - h / 2),
    ]
    vertices_lonlat = [twd97_to_lonlat(x, y) for x, y in vertices_3826]
    min_lon = min(p[0] for p in vertices_lonlat)
    max_lon = max(p[0] for p in vertices_lonlat)
    min_lat = min(p[1] for p in vertices_lonlat)
    max_lat = max(p[1] for p in vertices_lonlat)

    return {
        "id": grid_id,
        "vertices_3826": vertices_3826,
        "vertices_lonlat": vertices_lonlat,
        "bbox_lonlat": (min_lon, min_lat, max_lon, max_lat),
    }


def download_bbox_image(
    bbox_lonlat: tuple[float, float, float, float],
    zoom: int = 18,
    layer: str = "PHOTO2",
) -> tuple[Image.Image, tuple[float, float]]:
    if not (0 <= zoom <= 19):
        raise ValueError("NLSC PHOTO2 max zoom is 19; use a zoom between 0 and 19.")

    min_lon, min_lat, max_lon, max_lat = bbox_lonlat
    left_px, top_px = lonlat_to_pixel(min_lon, max_lat, zoom)
    right_px, bottom_px = lonlat_to_pixel(max_lon, min_lat, zoom)

    min_tile_x = math.floor(left_px / TILE_SIZE)
    max_tile_x = math.floor((right_px - 1) / TILE_SIZE)
    min_tile_y = math.floor(top_px / TILE_SIZE)
    max_tile_y = math.floor((bottom_px - 1) / TILE_SIZE)

    mosaic = Image.new(
        "RGB",
        ((max_tile_x - min_tile_x + 1) * TILE_SIZE, (max_tile_y - min_tile_y + 1) * TILE_SIZE),
    )
    for tile_x in range(min_tile_x, max_tile_x + 1):
        for tile_y in range(min_tile_y, max_tile_y + 1):
            tile = fetch_tile(tile_x, tile_y, zoom, layer=layer)
            mosaic.paste(tile, ((tile_x - min_tile_x) * TILE_SIZE, (tile_y - min_tile_y) * TILE_SIZE))

    crop_left = int(math.floor(left_px - min_tile_x * TILE_SIZE))
    crop_top = int(math.floor(top_px - min_tile_y * TILE_SIZE))
    crop_right = int(math.ceil(right_px - min_tile_x * TILE_SIZE))
    crop_bottom = int(math.ceil(bottom_px - min_tile_y * TILE_SIZE))
    image = mosaic.crop((crop_left, crop_top, crop_right, crop_bottom))
    return image, (left_px, top_px)


def apply_hex_mask(
    image: Image.Image,
    vertices_lonlat: list[tuple[float, float]],
    origin_pixel: tuple[float, float],
    zoom: int,
) -> Image.Image:
    origin_x, origin_y = origin_pixel
    polygon_px = []
    for lon, lat in vertices_lonlat:
        px, py = lonlat_to_pixel(lon, lat, zoom)
        polygon_px.append((px - origin_x, py - origin_y))

    mask = Image.new("L", image.size, 0)
    draw = ImageDraw.Draw(mask)
    draw.polygon(polygon_px, fill=255)

    result = image.convert("RGBA")
    result.putalpha(mask)
    return result


def download_hex_satellite_image(
    row: pd.Series,
    output_dir: str | Path = OUTPUT_DIR,
    zoom: int = 18,
    layer: str = "PHOTO2",
) -> Path:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    hex_geometry = get_hex_geometry(row)
    bbox_image, origin_pixel = download_bbox_image(hex_geometry["bbox_lonlat"], zoom=zoom, layer=layer)
    hex_image = apply_hex_mask(bbox_image, hex_geometry["vertices_lonlat"], origin_pixel, zoom=zoom)

    output_path = output_dir / f"{hex_geometry['id']}.png"
    hex_image.save(output_path)
    return output_path


def download_hex_satellite_images(
    grid_df: pd.DataFrame,
    output_dir: str | Path = OUTPUT_DIR,
    zoom: int = 18,
    layer: str = "PHOTO2",
    limit: int | None = None,
) -> pd.DataFrame:
    records = []
    rows = grid_df.head(limit) if limit is not None else grid_df
    for index, row in rows.iterrows():
        output_path = download_hex_satellite_image(row, output_dir=output_dir, zoom=zoom, layer=layer)
        records.append({"id": int(row["id"]), "save_path": str(output_path)})
        print(f"[{len(records)}/{len(rows)}] Saved {output_path.name}")
    return pd.DataFrame(records)


grid_df = load_hex_grid()

# ???????????? id?
TARGET_ROW_INDEX = 0
TARGET_ZOOM = 18
single_output_path = download_hex_satellite_image(
    grid_df.iloc[TARGET_ROW_INDEX],
    output_dir=OUTPUT_DIR,
    zoom=TARGET_ZOOM,
)
single_output_path




,X,Y,id,left,top,right,bottom,row_index,col_index,fid,VILLCODE,COUNTYNAME,TOWNNAME,VILLNAME,VILLENG,COUNTYID,COUNTYCODE,TOWNID,TOWNCODE,NOTE
0,120.323629,23.130739,163914,180152.131280,2.559421e+06,181306.831819,2.558421e+06,371,291,1.0,6.700019e+10,臺南市,善化區,嘉南里,Jianan Vil.,D,67000.0,D27,67000190.0,NaN
1,120.332062,23.135290,164476,181018.156684,2.559921e+06,182172.857222,2.558921e+06,371,292,1.0,6.700019e+10,臺南市,善化區,嘉南里,Jianan Vil.,D,67000.0,D27,67000190.0,NaN
2,120.332107,23.126261,164477,181018.156684,2.558921e+06,182172.857222,2.557921e+06,372,292,1.0,6.700019e+10,臺南市,善化區,嘉南里,Jianan Vil.,D,67000.0,D27,67000190.0,NaN
3,120.332196,23.108201,164479,181018.156684,2.556921e+06,182172.857222,2.555921e+06,374,292,1.0,6.700019e+10,臺南市,善化區,嘉南里,Jianan Vil.,D,67000.0,D27,67000190.0,NaN
4,120.340496,23.139841,165037,181884.182088,2.560421e+06,183038.882626,2.559421e+06,370,293,2.0,6.700019e+10,臺南市,善化區,嘉北里,Jiabei Vil.,D,67000.0,D27,67000190.0,NaN


WindowsPath('satellite_downloads/163914.png')